# Gold Layer: Regional Infrastructure Metrics

This notebook creates business-ready analytics combining EV charging infrastructure and traditional fuel stations across NSW regions.

## Business Questions Answered:
- **Regional EV Readiness**: Which regions have the best EV charging infrastructure?
- **Infrastructure Gap Analysis**: Where are EV charging deserts?
- **Fuel Price Comparison**: How do traditional fuel costs vary by region?
- **Transition Metrics**: EV vs traditional fuel infrastructure ratios

## Regional Segmentation:
NSW divided into 8 regions based on coordinates:
- **Sydney Metro**: Greater Sydney area
- **Central Coast**: Between Sydney and Newcastle
- **Hunter**: Newcastle and surrounds
- **Illawarra**: Wollongong and south coast
- **Southern Tablelands**: Inland south
- **Western NSW**: Far west regions
- **Northern NSW**: North coast and tablelands
- **Riverina**: Inland riverina

## Gold Table Output:
`mobility_ai.gold.regional_infrastructure_metrics`

## Key Metrics:
- EV charging station count, total capacity (kW), speed distribution
- Fuel station count, average prices by fuel type
- Infrastructure density (stations per 1000 km²)
- EV readiness score

In [0]:
# Configuration
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Table paths
EV_STATIONS_TABLE = "mobility_ai.silver.ev_charging_stations"
FUEL_PRICES_TABLE = "mobility_ai.silver.fuel_prices"
GOLD_TABLE = "mobility_ai.gold.regional_infrastructure_metrics"

# NSW Regional boundaries (approximate lat/lon)
REGIONS = [
    # (region_name, lat_min, lat_max, lon_min, lon_max)
    ("Sydney Metro", -34.2, -33.4, 150.5, 151.3),
    ("Central Coast", -33.6, -33.1, 151.2, 151.6),
    ("Hunter", -33.1, -32.5, 151.4, 152.0),
    ("Illawarra", -35.0, -34.2, 150.6, 151.0),
    ("Southern Tablelands", -36.5, -34.5, 148.5, 150.0),
    ("Western NSW", -33.5, -28.5, 140.0, 147.0),
    ("Northern NSW", -30.5, -28.0, 152.5, 154.0),
    ("Riverina", -36.0, -33.5, 144.0, 148.5)
]

print("\n" + "═" * 80)
print("⚙️  CONFIGURATION")
print("═" * 80)
print(f"\n📍 Source Tables:")
print(f"   • EV Stations:  {EV_STATIONS_TABLE}")
print(f"   • Fuel Prices:  {FUEL_PRICES_TABLE}")
print(f"\n🎯 Target Table:")
print(f"   • Gold Layer:   {GOLD_TABLE}")
print(f"\n🗺️  Regional Segmentation:")
print(f"   • Total Regions Defined: {len(REGIONS)}")
print("═" * 80 + "\n")

In [0]:
# Load silver tables
ev_stations = spark.table(EV_STATIONS_TABLE)
fuel_prices = spark.table(FUEL_PRICES_TABLE)

ev_count = ev_stations.count()
fuel_count = fuel_prices.count()

print("\n" + "═" * 80)
print("📊 DATA LOADING")
print("═" * 80)
print(f"\n✅ Silver Tables Loaded:")
print(f"   • EV Charging Stations: {ev_count:,} records")
print(f"   • Fuel Prices:          {fuel_count:,} records")
print("\n" + "─" * 80)
print("\n📋 EV Stations Schema:")
ev_stations.printSchema()
print("\n" + "─" * 80)
print("\n📋 Fuel Prices Schema:")
fuel_prices.printSchema()
print("═" * 80 + "\n")

In [0]:
# Create region mapping function using case/when for each region
def assign_region(lat_col, lon_col):
    region_expr = None
    
    for region_name, lat_min, lat_max, lon_min, lon_max in REGIONS:
        condition = (
            (F.col(lat_col).between(lat_min, lat_max)) &
            (F.col(lon_col).between(lon_min, lon_max))
        )
        if region_expr is None:
            region_expr = F.when(condition, region_name)
        else:
            region_expr = region_expr.when(condition, region_name)
    
    return region_expr.otherwise("Other")

# Assign regions to EV stations
ev_with_region = ev_stations.withColumn(
    "region",
    assign_region("latitude", "longitude")
)

print("\n" + "═" * 80)
print("🗺️  REGIONAL ASSIGNMENT")
print("═" * 80)
print("\n⚡ EV Charging Stations by Region:")
print("─" * 80)
ev_with_region.groupBy("region").count().orderBy("count", ascending=False).show(truncate=False)

print("\n⛽ Fuel Station Location Data:")
print("─" * 80)
print("   ℹ️  Note: Fuel price data does not include geographic coordinates.")
print("   📊 Fuel metrics will be calculated at NSW state level.")
print("═" * 80 + "\n")

In [0]:
# Aggregate EV charging metrics by region
ev_metrics = ev_with_region.groupBy("region").agg(
    # Station counts
    F.count("*").alias("ev_station_count"),
    
    # Valid data percentages
    F.sum(F.when(F.col("has_valid_location"), 1).otherwise(0)).alias("valid_locations"),
    F.sum(F.when(F.col("has_valid_rating"), 1).otherwise(0)).alias("valid_ratings"),
    
    # Charging capacity metrics (only for valid ratings)
    F.sum(F.when(F.col("has_valid_rating"), F.col("charger_rating_kw")).otherwise(0)).alias("total_capacity_kw"),
    F.avg(F.when(F.col("has_valid_rating"), F.col("charger_rating_kw"))).alias("avg_capacity_kw"),
    F.max(F.col("charger_rating_kw")).alias("max_capacity_kw"),
    
    # Speed category breakdown
    F.sum(F.when(F.col("charging_speed") == "Slow", 1).otherwise(0)).alias("slow_chargers"),
    F.sum(F.when(F.col("charging_speed") == "Fast", 1).otherwise(0)).alias("fast_chargers"),
    F.sum(F.when(F.col("charging_speed") == "Rapid", 1).otherwise(0)).alias("rapid_chargers"),
    F.sum(F.when(F.col("charging_speed") == "Ultra-Rapid", 1).otherwise(0)).alias("ultra_rapid_chargers"),
    F.sum(F.when(F.col("charging_speed") == "Unknown", 1).otherwise(0)).alias("unknown_speed_chargers")
)

print("\n" + "═" * 80)
print("⚡ EV INFRASTRUCTURE AGGREGATION")
print("═" * 80)
print("\n📊 Regional Metrics Summary:")
print("─" * 80)
ev_metrics.orderBy("ev_station_count", ascending=False).show(truncate=False)
print("═" * 80 + "\n")

In [0]:
# Aggregate fuel station metrics at STATE level (no location data available)
# These will be joined as reference data to all regions

fuel_metrics_statewide = fuel_prices.agg(
    # Station counts (distinct to avoid counting same station multiple times)
    F.countDistinct("stationcode").alias("fuel_station_count_nsw"),
    
    # Valid data counts
    F.sum(F.when(F.col("has_valid_price"), 1).otherwise(0)).alias("valid_prices_nsw"),
    
    # Price metrics by fuel category (only valid prices)
    F.avg(
        F.when(
            (F.col("fuel_category") == "Regular Unleaded") & (F.col("has_valid_price")),
            F.col("price")
        )
    ).alias("avg_regular_unleaded_price"),
    
    F.avg(
        F.when(
            (F.col("fuel_category") == "Diesel") & (F.col("has_valid_price")),
            F.col("price")
        )
    ).alias("avg_diesel_price"),
    
    F.avg(
        F.when(
            (F.col("fuel_category") == "LPG") & (F.col("has_valid_price")),
            F.col("price")
        )
    ).alias("avg_lpg_price"),
    
    # Fuel type availability (count distinct stations offering each type)
    F.countDistinct(
        F.when(F.col("fuel_category") == "Regular Unleaded", F.col("stationcode"))
    ).alias("stations_with_unleaded_nsw"),
    
    F.countDistinct(
        F.when(F.col("fuel_category") == "Diesel", F.col("stationcode"))
    ).alias("stations_with_diesel_nsw"),
    
    F.countDistinct(
        F.when(F.col("fuel_category") == "LPG", F.col("stationcode"))
    ).alias("stations_with_lpg_nsw")
)

print("\n" + "═" * 80)
print("⛽ FUEL INFRASTRUCTURE AGGREGATION (NSW-WIDE)")
print("═" * 80)
print("\n📊 State-Wide Fuel Metrics:")
print("─" * 80)
fuel_metrics_statewide.show(truncate=False)
print("\n   ℹ️  Note: Fuel station counts represent NSW-wide totals.")
print("   📍 Regional breakdown unavailable (no coordinates in source data).")
print("═" * 80 + "\n")

In [0]:
# Cross join to add state-wide fuel metrics to each region
# (Since fuel data lacks location, we add reference metrics to all regions)
gold_df = ev_metrics.crossJoin(fuel_metrics_statewide)

# Fill nulls for regions that might not have EV infrastructure
gold_df = gold_df.fillna(0, subset=[
    "ev_station_count",
    "total_capacity_kw", "avg_capacity_kw",
    "slow_chargers", "fast_chargers", "rapid_chargers", "ultra_rapid_chargers"
])

# Calculate EV Readiness Score (0-100)
# Based on: station count (40%), fast/rapid chargers (30%), total capacity (30%)
gold_df = gold_df.withColumn(
    "ev_readiness_score",
    F.least(
        F.lit(100),
        (
            # Station count component (capped at 40 points, 1 point per 5 stations)
            F.least(F.lit(40), F.col("ev_station_count") / 5) +
            # Fast charger availability (30 points for having 50%+ fast/rapid/ultra-rapid)
            F.when(
                F.col("ev_station_count") > 0,
                F.least(
                    F.lit(30),
                    ((F.col("fast_chargers") + F.col("rapid_chargers") + F.col("ultra_rapid_chargers")) / 
                     F.col("ev_station_count") * 30)
                )
            ).otherwise(0) +
            # Capacity component (30 points for 1000+ kW total)
            F.least(F.lit(30), F.col("total_capacity_kw") / 1000 * 30)
        )
    ).cast("int")
)

# Add processing metadata
gold_df = gold_df.withColumn("gold_processed_at", F.current_timestamp())
gold_df = gold_df.withColumn("reporting_date", F.current_date())

total_regions = gold_df.count()

print("\n" + "═" * 80)
print("🏆 GOLD LAYER METRICS CALCULATION")
print("═" * 80)
print(f"\n📊 Combined Regional Infrastructure Metrics ({total_regions} regions):")
print("─" * 80)
gold_df.select(
    "region",
    "ev_readiness_score",
    "ev_station_count",
    "total_capacity_kw",
    "rapid_chargers",
    "ultra_rapid_chargers",
    "fuel_station_count_nsw"
).orderBy("ev_readiness_score", ascending=False).show(truncate=False)

print("\n   ℹ️  EV Readiness Score Algorithm:")
print("      • Station Count:    40% (1 point per 5 stations, max 40)")
print("      • Fast Chargers:    30% (based on % of fast/rapid/ultra-rapid)")
print("      • Total Capacity:   30% (1 point per 33.3 kW, max 30)")
print("\n   ℹ️  fuel_station_count_nsw: NSW-wide reference total (same for all regions)")
print("═" * 80 + "\n")

In [0]:
# Write to gold table
record_count = gold_df.count()

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_TABLE)

print("\n" + "═" * 80)
print("💾 GOLD TABLE PERSISTENCE")
print("═" * 80)
print(f"\n✅ Success! Written {record_count} regional records to:")
print(f"   📊 {GOLD_TABLE}")
print("\n   📝 Write Mode:    OVERWRITE")
print("   📄 Format:        Delta Lake")
print("   🔄 Schema:        Auto-evolved")
print("═" * 80 + "\n")

In [0]:
# Load and validate gold table
gold_validation = spark.table(GOLD_TABLE)

region_count = gold_validation.count()
reporting_date = gold_validation.select('reporting_date').first()[0]

print("\n" + "═" * 80)
print("🎯 GOLD LAYER: REGIONAL INFRASTRUCTURE ANALYSIS")
print("═" * 80)
print(f"\n📅 Reporting Date:    {reporting_date}")
print(f"🗺️  Total Regions:     {region_count}")
print(f"📊 Table:             {GOLD_TABLE}")
print("\n" + "═" * 80)

# Top regions by EV readiness
print("\n🏆 TOP 5 REGIONS BY EV READINESS SCORE")
print("─" * 80)
gold_validation.select(
    "region",
    "ev_readiness_score",
    "ev_station_count",
    "total_capacity_kw",
    "rapid_chargers",
    "ultra_rapid_chargers"
).orderBy("ev_readiness_score", ascending=False).show(5, truncate=False)

# EV infrastructure by region
print("\n⚡ EV INFRASTRUCTURE BY REGION")
print("─" * 80)
gold_validation.select(
    "region",
    "ev_station_count",
    "slow_chargers",
    "fast_chargers",
    "rapid_chargers",
    "ultra_rapid_chargers"
).orderBy("ev_station_count", ascending=False).show(truncate=False)

# Charging capacity summary
print("\n⚡ TOTAL CHARGING CAPACITY BY REGION (kW)")
print("─" * 80)
gold_validation.select(
    "region",
    F.round("total_capacity_kw", 1).alias("total_kw"),
    F.round("avg_capacity_kw", 1).alias("avg_kw"),
    F.round("max_capacity_kw", 1).alias("max_kw")
).orderBy("total_kw", ascending=False).show(truncate=False)

# Fuel price summary (state-wide)
print("\n⛽ NSW STATE-WIDE FUEL PRICES (cents/liter)")
print("─" * 80)
gold_validation.select(
    F.round("avg_regular_unleaded_price", 1).alias("unleaded"),
    F.round("avg_diesel_price", 1).alias("diesel"),
    F.round("avg_lpg_price", 1).alias("lpg"),
    "fuel_station_count_nsw"
).limit(1).show(truncate=False)
print("   ℹ️  Note: Fuel prices are NSW-wide averages (no regional data available)\n")

# Key insights summary
print("═" * 80)
print("💡 KEY INSIGHTS")
print("═" * 80)

total_ev = gold_validation.agg(F.sum("ev_station_count")).first()[0]
total_capacity = gold_validation.agg(F.sum("total_capacity_kw")).first()[0]
total_fuel_nsw = gold_validation.select("fuel_station_count_nsw").first()[0]
ev_ratio = total_ev/(total_ev+total_fuel_nsw)*100

print("\n📊 NSW Infrastructure Overview:")
print(f"   • Total EV Charging Stations:  {int(total_ev):,}")
print(f"   • Total Fuel Stations (NSW):   {int(total_fuel_nsw):,}")
print(f"   • EV Infrastructure Ratio:     {ev_ratio:.1f}%")
print(f"   • Total EV Charging Capacity:  {total_capacity:,.0f} kW")

top_region = gold_validation.orderBy("ev_readiness_score", ascending=False).first()
bottom_region = gold_validation.orderBy("ev_readiness_score", ascending=True).first()

print("\n🏆 Regional Leaders:")
print(f"   • Most EV-Ready:   {top_region['region']:20s} (Score: {top_region['ev_readiness_score']}/100)")
print(f"   • Least EV-Ready:  {bottom_region['region']:20s} (Score: {bottom_region['ev_readiness_score']}/100)")

print("\n" + "═" * 80)
print("✅ GOLD LAYER ANALYSIS COMPLETE")
print("═" * 80 + "\n")